# Cell 1 — Setup:


In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

VOL    = "/Volumes/workspace/default/maplebank"
SILVER = f"{VOL}/silver"
GOLD   = f"{VOL}/gold"
TXN    = f"{SILVER}/transactions"

print(f"Current Silver txn count: {spark.read.format('delta').load(TXN).count():,}")

Current Silver txn count: 10,000


## Cell 2 — Read the ledger (DESCRIBE HISTORY):

In [0]:
spark.sql(f"DESCRIBE HISTORY delta.`{TXN}`") \
     .select("version", "timestamp", "operation", "operationParameters") \
     .show(10, truncate=False)

+-------+-------------------+---------+------------------------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                                           |
+-------+-------------------+---------+------------------------------------------------------------------------------+
|0      |2026-06-11 13:13:17|WRITE    |{mode -> Overwrite, statsOnLoad -> false, partitionBy -> ["transaction_date"]}|
+-------+-------------------+---------+------------------------------------------------------------------------------+



# Cell 3 — Time travel:

In [0]:
# Append a small "late-arriving" batch to create a new version
late_batch = spark.read.format("delta").load(TXN).limit(5) \
    .withColumn("transaction_id", F.concat(F.lit("LATE_"), F.col("transaction_id")))

late_batch.write.format("delta").mode("append").save(TXN)

current = spark.read.format("delta").load(TXN).count()

# Now travel back to version 0 — before the append
v0 = spark.read.format("delta").option("versionAsOf", 0).load(TXN).count()

print(f"Version 0 (as originally built): {v0:,}")
print(f"Current version:                 {current:,}")
print(f"Difference: {current - v0} — the late batch, invisible in the past ✅")

Version 0 (as originally built): 10,000
Current version:                 10,005
Difference: 5 — the late batch, invisible in the past ✅


## Cell 4 — MERGE upsert (the exactly-once pattern):

In [0]:
# Simulate tonight's feed: 1 RESENT transaction (same ID, corrected amount)
#                        + 1 BRAND-NEW transaction
existing_id = spark.read.format("delta").load(TXN) \
    .filter(~F.col("transaction_id").startswith("LATE_")) \
    .select("transaction_id").first()[0]

resent = spark.read.format("delta").load(TXN) \
    .filter(F.col("transaction_id") == existing_id) \
    .withColumn("amount_cad", F.lit(7777.77))          # corrected amount

brand_new = resent.withColumn("transaction_id", F.lit("TXN_NEW_001")) \
                  .withColumn("amount_cad", F.lit(123.45))

incoming = resent.union(brand_new)

before = spark.read.format("delta").load(TXN).count()

DeltaTable.forPath(spark, TXN).alias("t") \
    .merge(incoming.alias("s"), "t.transaction_id = s.transaction_id") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

after = spark.read.format("delta").load(TXN).count()
print(f"Before: {before:,}  After: {after:,}  (+1 insert, 1 update, 0 duplicates)")

spark.read.format("delta").load(TXN) \
    .filter(F.col("transaction_id").isin(existing_id, "TXN_NEW_001")) \
    .select("transaction_id", "amount_cad").show()

Before: 10,005  After: 10,006  (+1 insert, 1 update, 0 duplicates)
+--------------+----------+
|transaction_id|amount_cad|
+--------------+----------+
|    TXN1000690|   7777.77|
|   TXN_NEW_001|    123.45|
+--------------+----------+



## Cell 5 — SCD Type 2: the customer who moved:

In [0]:
SCD2 = f"{GOLD}/dim_customer_scd2"

# 5a. Initialize the SCD2 dimension from Silver (first load = all current)
spark.read.format("delta").load(f"{SILVER}/customers") \
    .select("customer_id", "first_name", "last_name", "city", "province", "postal_code_fsa") \
    .withColumn("is_current", F.lit(True)) \
    .withColumn("start_date", F.current_date()) \
    .withColumn("end_date",   F.lit(None).cast("date")) \
    .write.format("delta").mode("overwrite").save(SCD2)

# 5b. The change event: a Toronto customer moves to Mississauga
mover_id = spark.read.format("delta").load(SCD2) \
    .filter(F.col("city") == "Toronto").select("customer_id").first()[0]

update = spark.read.format("delta").load(SCD2) \
    .filter(F.col("customer_id") == mover_id) \
    .withColumn("city",            F.lit("Mississauga")) \
    .withColumn("postal_code_fsa", F.lit("L5B"))

# 5c. Expire the old current row...
DeltaTable.forPath(spark, SCD2).alias("t") \
    .merge(update.alias("s"),
           "t.customer_id = s.customer_id AND t.is_current = true") \
    .whenMatchedUpdate(
        condition="t.city <> s.city",
        set={"is_current": "false", "end_date": "current_date()"}) \
    .execute()

# 5d. ...then insert the new current row
update.withColumn("is_current", F.lit(True)) \
      .withColumn("start_date", F.current_date()) \
      .withColumn("end_date",   F.lit(None).cast("date")) \
      .write.format("delta").mode("append").save(SCD2)

# 5e. The proof: TWO rows for one customer — full history preserved
spark.read.format("delta").load(SCD2) \
    .filter(F.col("customer_id") == mover_id) \
    .select("customer_id", "city", "postal_code_fsa",
            "is_current", "start_date", "end_date") \
    .show(truncate=False)

+-----------+-----------+---------------+----------+----------+----------+
|customer_id|city       |postal_code_fsa|is_current|start_date|end_date  |
+-----------+-----------+---------------+----------+----------+----------+
|CUST100153 |Toronto    |K1S            |false     |2026-06-11|2026-06-11|
|CUST100153 |Mississauga|L5B            |true      |2026-06-11|NULL      |
+-----------+-----------+---------------+----------+----------+----------+



## Cell 6 — OPTIMIZE + VACUUM (dry run only!):

In [0]:
# Compact small files + co-locate by customer_id for fast lookups
spark.sql(f"OPTIMIZE delta.`{TXN}` ZORDER BY (customer_id)")

# The OPTIMIZE now appears in the ledger too:
spark.sql(f"DESCRIBE HISTORY delta.`{TXN}`") \
     .select("version", "operation").show(5)

# VACUUM dry run — see what WOULD be deleted, without deleting
spark.sql(f"VACUUM delta.`{TXN}` DRY RUN").show(5, truncate=False)
print("⚠ We do NOT run real VACUUM — it would erase time-travel history.")
print("  Banking rule: VACUUM retention is a compliance decision, not an engineering one.")

+-------+---------+
|version|operation|
+-------+---------+
|      3| OPTIMIZE|
|      2|    MERGE|
|      1|    WRITE|
|      0|    WRITE|
+-------+---------+

+----+
|path|
+----+
+----+

⚠ We do NOT run real VACUUM — it would erase time-travel history.
  Banking rule: VACUUM retention is a compliance decision, not an engineering one.
